# 02 Training

This notebook trains LSTM, CNN-LSTM, and TFT across walk-forward folds using the project trainer.

In [ ]:
# Step 1: load configuration and set up deterministic behavior.
from src.utils.config import load_config, set_seed, resolve_device
from src.pipeline import prepare_engineered_universe
from src.data.dataset import create_all_fold_dataloaders
from src.training.trainer import train_model_across_folds, pairwise_mcnemar_across_models

config = load_config('config/default.yaml')
set_seed(config['experiment']['random_seed'])
device = resolve_device(config['experiment']['device'])
device

In [ ]:
# Step 2: build engineered and labeled modeling dataframe.
modeling_df, feature_cols = prepare_engineered_universe(config=config, force_refresh=False)
len(modeling_df), len(feature_cols)

In [ ]:
# Step 3: create fold-specific dataloaders with leakage-safe normalization.
fold_bundles = create_all_fold_dataloaders(modeling_df, feature_cols, config)
len(fold_bundles)

In [ ]:
# Step 4: train each architecture across all folds.
outputs = {}
for model_name in ['lstm', 'cnn_lstm', 'tft']:
    outputs[model_name] = train_model_across_folds(model_name, fold_bundles, config, device)

{name: payload['summary'] for name, payload in outputs.items()}

In [ ]:
# Step 5: run pairwise McNemar tests for statistical significance checks.
mcnemar = pairwise_mcnemar_across_models(outputs)
mcnemar